# 第二阶段

## 一、模型注册

In [ ]:
#1. 编写脚本
-- 详见：/home/mlops/quant-mlops-project/src/registry/register.py

In [ ]:
#2. 执行脚本,找到test_rank-ic最优的fold 的 run_id 
(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python -m src.registry.register --run-id c5dcfd5860494241b79738eed6902719
run_id       : c5dcfd5860494241b79738eed6902719
batch_id     : 20261012-153053
fold         : 1
test_ic      : 0.0158
test_rank_ic : 0.0269
model_uri    : models:/m-c99427e69fcd4bbe810496e0fb962c42
Successfully registered model 'us-stock-lgbm-1d'.
Created version '1' of model 'us-stock-lgbm-1d'.
registered: us-stock-lgbm-1d version 1
alias 'staging' set

In [ ]:
# 3. 再次训练，找到test_rank-ic最优的fold 的 run_id  ，然后可以再次注册，版本就成为2了，如下：
(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python -m src.registry.register --run-id 167526a7bad6484794e06a3718fc6306
run_id       : 167526a7bad6484794e06a3718fc6306
batch_id     : 20261012-162409
fold         : 1
test_ic      : 0.0158
test_rank_ic : 0.0269
model_uri    : models:/m-7279610f66694d0bb1f6865ee9723b12
Registered model 'us-stock-lgbm-1d' already exists. Creating a new version of this model...
Created version '2' of model 'us-stock-lgbm-1d'.
registered: us-stock-lgbm-1d version 2
alias 'staging' set

### 注意：register.py中，使用了alias功能，配置为：candidate，模型的真实状态，可以再UI中，手动维护，初始注册时，别名先配置为了 candidate
————————————————————————————————————
### 特别说明：
- MLflow 从 2.9.0 版本开始就废弃了 Model Registry 的 Stage 概念（Staging、Production、Archived），并计划在未来版本中移除相关 API。
- 官方的迁移指南明确写道，替代方案是 Model Version Aliases（别名）和 Tags（标签）。
- 可以考虑使用：@candidate（候选者）  @champion（当前主模型）或 @challenger（挑战者）这类别名 

# 二、模型服务（FastAPI）

### 目标：把注册的模型包成 REST API，Docker 化，能被外部调用。
### 接口：
- GET /health — 健康检查

- POST /predict — 输入 ticker + date，返回预测值

- GET /model/info — 返回当前加载的模型版本、alias、加载时间

- 关键点：

- 启动时从 models:/us-stock-lgbm-1d@champion（或 @candidate）加载模型

- 预测时需要先算特征——这里有个坑：特征计算需要历史数据，不是单点输入能算的。解决方式：服务启动时预加载 features.parquet，/predict 按 ticker+date 查表

- 加基础的输入校验（date 格式、ticker 是否在池子里）

### 1. 安装依赖
uv add fastapi uvicorn

### 2. 写 model_loader.py：加载模型 + 特征表
- 详见 ：quant-mlops-project/src/serving/model_loader.py

## 特别注意：由于fold1虽然是fold1-5中的champion，它的模型指标最好，但是fold1的训练数据量是fold5的一半，我认为需要进行如下调整
## 在最终的测试场景
将version2 的模型，阶段改为：@champion  --使用UI方式修改，不用命令行
将fold5，注册为version3，阶段为：@candidate   --register.py 代码中，默认使用 candidate  

### 3. 写 schemas.py  ：股票/金融预测模型服务的接口定义。
- 这段代码是用 Pydantic 定义的几个数据模型（BaseModel）
- 用来定义 API 的请求体和响应体结构。
- 详见：quant-mlops-project/src/serving/schemas.py

### 4. 写app.py: 基于 FastAPI 的量化模型推理服务（Quant MLOps Serving）。
- 它把之前定义好的 Pydantic 模型和实际的业务逻辑串起来，对外提供三个 HTTP 接口。
- 详见：/home/mlops/quant-mlops-project/src/serving/app.py

In [ ]:
# 启动服务，进行本地测试
uv run uvicorn src.serving.app:app --reload --port 8000

In [ ]:
# 健康检查
curl http://localhost:8000/health

# 模型信息
curl http://localhost:8000/model/info

# 预测（用 fold 5 的未见区间数据，2024 年）
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"ticker": "AAPL", "date": "2024-06-03"}'